# Laboratorio 01 — SQL básico sobre tu propio dataset

**Semana:** 03 | **Actividad de referencia:** Actividad 01  
**Modalidad:** Individual | **Entorno:** Databricks (Unity Catalog)

---

## Instrucciones generales

Aplica las consultas SQL aprendidas en la Actividad 01 sobre un dataset **de tu elección**. El dataset debe cargarse como tabla Delta en Unity Catalog para poder consultarlo con Spark SQL.

## Parte 1 — Descripción del dataset

Documenta tu dataset antes de escribir código:

1. **Nombre y fuente:** ¿Cómo se llama y de dónde lo obtuviste? (incluye URL)
2. **Dominio:** ¿Qué área o problema describe?
3. **¿Por qué lo elegiste?** ¿Qué pregunta de negocio quieres responder con SQL?
4. **Preguntas de negocio:** Lista al menos 3 preguntas que responderás con SELECT, GROUP BY y HAVING.

**Escribe tu respuesta aquí:**

## Parte 2 — Cargar el dataset como tabla Delta

Carga el archivo desde tu volumen y guárdalo como tabla Delta en el esquema `default` (o uno propio). A partir de aquí trabajarás exclusivamente con SQL.

In [ ]:
# Ajusta VOL, ARCHIVO y TABLA_NOMBRE
VOL          = "/Volumes/workspace/default/week_3"  # cambia si usas otro volumen
ARCHIVO      = "tu_archivo.csv"                      # nombre real del archivo
TABLA_NOMBRE = "workspace.default.lab03_01_mi_dataset"  # nombre de la tabla Delta

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(f"{VOL}/{ARCHIVO}")

df.write.format("delta").mode("overwrite").saveAsTable(TABLA_NOMBRE)
print(f"✓ Tabla creada: {TABLA_NOMBRE}")
print(f"  Filas: {df.count():,} | Columnas: {len(df.columns)}")

## Parte 3 — Perfil técnico con SQL

Explora la estructura de la tabla usando únicamente Spark SQL. Completa cada celda y añade tus observaciones en markdown.

In [ ]:
# Schema y metadatos de la tabla
# Sustituye el nombre si cambiaste TABLA_NOMBRE
spark.sql("DESCRIBE TABLE EXTENDED workspace.default.lab03_01_mi_dataset").show(50, truncate=False)

In [ ]:
# Número de registros
spark.sql("SELECT COUNT(*) AS total_registros FROM workspace.default.lab03_01_mi_dataset").show()

In [ ]:
# Primeros 10 registros
spark.sql("SELECT * FROM workspace.default.lab03_01_mi_dataset LIMIT 10").show(truncate=False)

In [ ]:
# Porcentaje de nulos por columna
# Ajusta la lista de columnas según tu dataset
# Patrón: SUM(CASE WHEN col IS NULL OR col = '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)
spark.sql("""
    SELECT
        COUNT(*) AS total
        -- Añade una línea por cada columna de tu dataset:
        -- , ROUND(SUM(CASE WHEN columna1 IS NULL OR columna1 = '' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_nulos_columna1
        -- , ROUND(SUM(CASE WHEN columna2 IS NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_nulos_columna2
    FROM workspace.default.lab03_01_mi_dataset
""").show(truncate=False)

**Observaciones de nulos:** ¿Cuáles columnas tienen mayor porcentaje? ¿Qué implicaría filtrarlas con `WHERE col IS NOT NULL`?

In [ ]:
# Cardinalidad de columnas categóricas
# Reemplaza 'columna_categorica' por el nombre real de cada columna string de tu dataset
spark.sql("""
    SELECT
        'columna_categorica' AS columna,
        COUNT(DISTINCT columna_categorica) AS valores_unicos
    FROM workspace.default.lab03_01_mi_dataset
    -- UNION ALL
    -- SELECT 'otra_columna', COUNT(DISTINCT otra_columna) FROM workspace.default.lab03_01_mi_dataset
""").show()

**Observaciones de cardinalidad:** ¿Qué columnas tienen baja cardinalidad (buenos candidatos para GROUP BY)? ¿Cuáles tienen alta cardinalidad (posibles IDs)?

## Parte 4 — Aplicar SQL de la Actividad 01

Implementa al menos **5 consultas** usando los conceptos de la Actividad 01. Para cada una añade una celda markdown con tu interpretación del resultado.

In [ ]:
# Consulta 1 — SELECT con WHERE y ORDER BY
spark.sql("""
    SELECT *
    FROM workspace.default.lab03_01_mi_dataset
    WHERE -- tu condición
    ORDER BY -- tu columna
    LIMIT 20
""").show(truncate=False)

**Conclusión consulta 1:**

In [ ]:
# Consulta 2 — GROUP BY con COUNT, SUM o AVG
spark.sql("""
    SELECT
        -- columna_categoria,
        COUNT(*) AS total,
        -- SUM(columna_numerica) AS suma,
        AVG(-- columna_numerica --) AS promedio
    FROM workspace.default.lab03_01_mi_dataset
    GROUP BY -- columna_categoria
    ORDER BY total DESC
    LIMIT 15
""").show(truncate=False)

**Conclusión consulta 2:**

In [ ]:
# Consulta 3 — HAVING para filtrar grupos
spark.sql("""
    SELECT
        -- columna_categoria,
        COUNT(*) AS total
    FROM workspace.default.lab03_01_mi_dataset
    GROUP BY -- columna_categoria
    HAVING COUNT(*) > -- tu umbral
    ORDER BY total DESC
""").show(truncate=False)

**Conclusión consulta 3:** ¿Qué diferencia hay entre filtrar con WHERE antes del GROUP BY y con HAVING después?

In [ ]:
# Consulta 4 — libre: aplica lo que más sentido tenga para tu dataset
spark.sql("""

""").show(truncate=False)

**Conclusión consulta 4:**

In [ ]:
# Consulta 5 — libre: la más compleja o creativa que puedas formular
spark.sql("""

""").show(truncate=False)

**Conclusión consulta 5:**

## Parte 5 — Preguntas de negocio

Responde las 3 preguntas que planteaste en la Parte 1 usando Spark SQL. Cada respuesta:
- Consulta SQL completa
- Celda markdown con la conclusión en lenguaje no técnico

In [ ]:
# Pregunta 1:
spark.sql("""

""").show(truncate=False)

**Conclusión pregunta 1:**

In [ ]:
# Pregunta 2:
spark.sql("""

""").show(truncate=False)

**Conclusión pregunta 2:**

In [ ]:
# Pregunta 3:
spark.sql("""

""").show(truncate=False)

**Conclusión pregunta 3:**

## Parte 6 — Reflexión final

1. ¿Qué consulta te resultó más difícil de formular? ¿Por qué?
2. ¿En qué se diferencia escribir la misma consulta en SQL vs PySpark (`groupBy`, `filter`, `agg`)?
3. ¿Cuándo preferirías SQL y cuándo PySpark para este tipo de análisis?
4. ¿Qué hallazgo te sorprendió más de tu dataset?

---

## Entrega en Git

```bash
git add semana_03/laboratorios/lab_01_sql_basico.ipynb
git commit -m "lab: semana03 lab01 sql basico <nombre-dataset> - <tu-nombre>"
git push origin feature/semana03-sql-<tu-nombre>
```